# Running Monte Carlo Transport Independently

This tutorial demonstrates how to run the Monte Carlo transport loop directly using `Simulation.from_config` without running full TARDIS iterations. This approach gives you direct control over the Monte Carlo transport process and tracker-based postprocessing.

**Topics covered:**
- Initializing simulation state manually
- Running transport with packet tracking enabled  
- Accessing tracker data for analysis
- Generating virtual packets from tracker spawn events

In [ ]:
from numba import config as nconfig

nconfig.DISABLE_JIT = False

In [ ]:
from pathlib import Path

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np

from tardis.io.atom_data import AtomData
from tardis.io.configuration.config_reader import Configuration
from tardis.simulation import Simulation
from tardis.transport.montecarlo.estimators.estimators_bulk import (
    init_estimators_bulk,
)
from tardis.transport.montecarlo.estimators.estimators_line import (
    init_estimators_line,
)
from tardis.transport.montecarlo.modes.classic.montecarlo_transport import (
    montecarlo_transport,
)
from tardis.transport.montecarlo.packet_source.black_body import (
    BlackBodySimpleSource,
)
from tardis.transport.montecarlo.packets.trackers import (
    trackers_full_to_df,
)
from tardis.spectrum.spectrum_solver import VirtualPacketSolver

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/tardis-sn/tardis/master/docs/tardis_example.yml

In [ ]:
CONFIG_FILE_NAME = "tardis_example.yml"
NUMBER_OF_PACKETS = 10000
NUMBER_OF_VPACKETS = 3  # Virtual packets per spawn event (for postprocessing)
SHOW_PROGRESS_BARS = True
ENABLE_RPACKET_TRACKING = True  # Enable full packet tracking for virtual packet generation

In [ ]:
# Setup simulation state from config
config_file = Path(CONFIG_FILE_NAME)
if not config_file.exists():
    raise FileNotFoundError(f"Configuration file {CONFIG_FILE_NAME} not found")

config = Configuration.from_yaml(str(config_file))
atom_data = AtomData.from_hdf("kurucz_cd23_chianti_H_He_latest.h5")
sim = Simulation.from_config(config, atom_data=atom_data)

print("Simulation created successfully!")

In [ ]:
# Initialize opacity and macro atom states manually
sim.opacity_state = sim.opacity.legacy_solve(sim.plasma)

if sim.macro_atom is not None:
    sim.macro_atom_state = sim.macro_atom.solve(
        sim.plasma.j_blues,
        sim.opacity_state.beta_sobolev,
        sim.plasma.stimulated_emission_factor,
    )
else:
    sim.macro_atom_state = None

print("Opacity and macro atom states initialized!")

In [ ]:
# Extract states from simulation
geometry_state = sim.simulation_state.geometry
opacity_state = sim.opacity_state
montecarlo_configuration = sim.transport.montecarlo_configuration
time_explosion = sim.simulation_state.time_explosion.to(u.s).value
spectrum_frequency_grid = sim.transport.spectrum_frequency_grid.to(u.Hz).value

# Create packet source (independent from sim.transport.packet_source)
packet_source = BlackBodySimpleSource(
    radius=geometry_state.r_inner_active[0],
    temperature=sim.simulation_state.t_inner,
    base_seed=23111963,
)

# Initialize estimators
n_lines_by_n_cells_tuple = opacity_state.tau_sobolev.shape
n_cells = geometry_state.no_of_shells

estimators_bulk = init_estimators_bulk(n_cells)
estimators_line = init_estimators_line(n_lines_by_n_cells_tuple)

# Convert to numba-compatible states
geometry_state_numba = geometry_state.to_numba()
line_interaction_type = montecarlo_configuration.LINE_INTERACTION_TYPE
opacity_state_numba = opacity_state.to_numba(
    sim.macro_atom_state, line_interaction_type
)

print("Monte Carlo states prepared!")

## Running Transport with Packet Tracking

When tracking is enabled, all spawn events (initial emission + line + electron scattering) are logged to `rpacket_trackers` for later postprocessing.

In [ ]:
# Generate tracker list for full tracking
if ENABLE_RPACKET_TRACKING:
    from tardis.transport.montecarlo.packets.trackers import generate_tracker_full_list
    
    rpacket_trackers = generate_tracker_full_list(
        NUMBER_OF_PACKETS,
        montecarlo_configuration.INITIAL_TRACKING_ARRAY_LENGTH,
    )
    print(f"Full RPacket tracking enabled (initial capacity: {montecarlo_configuration.INITIAL_TRACKING_ARRAY_LENGTH})")
else:
    from tardis.transport.montecarlo.packets.trackers.tracker_last_interaction_util import (
        generate_tracker_last_interaction_list,
    )
    
    rpacket_trackers = generate_tracker_last_interaction_list(NUMBER_OF_PACKETS)
    print("Last interaction tracking only")

# Create packet collection
seed_offset = 0
packet_collection = packet_source.create_packets(NUMBER_OF_PACKETS, seed_offset)

# Run Monte Carlo transport
# Note: Virtual packets are generated via postprocessing, not inline
(
    v_packets_energy_hist,
    vpacket_tracker,
    estimators_bulk_result,
    estimators_line_result,
) = montecarlo_transport(
    packet_collection,
    geometry_state_numba,
    time_explosion,
    opacity_state_numba,
    montecarlo_configuration,
    spectrum_frequency_grid,
    rpacket_trackers,
    NUMBER_OF_VPACKETS,  # This is just configuration; postprocessing generates actual vpackets
    SHOW_PROGRESS_BARS,
)

print("Monte Carlo transport completed successfully!")

## Accessing Tracker Data

The tracker captures spawn events during transport. Convert to DataFrame for analysis:

In [ ]:
if ENABLE_RPACKET_TRACKING:
    tracker_df = trackers_full_to_df(rpacket_trackers)
    print(f"Tracker captured {len(tracker_df):,} spawn events")
    print(f"Tracker columns: {list(tracker_df.columns)}")
    print(f"\nFirst 5 spawn events:")
    print(tracker_df.head())
else:
    from tardis.transport.montecarlo.packets.trackers.tracker_last_interaction_util import (
        trackers_last_interaction_to_df,
    )
    
    tracker_df = trackers_last_interaction_to_df(rpacket_trackers)
    print(f"Last interaction tracker: {len(tracker_df)} packets")
    print(tracker_df.head())

## Generating Virtual Packets from Tracker Data

Virtual packets are generated via postprocessing using `VirtualPacketSolver`, which extracts spawn events from the tracker DataFrame.

In [ ]:
if ENABLE_RPACKET_TRACKING and NUMBER_OF_VPACKETS > 0:
    # Create VirtualPacketSolver with necessary parameters
    vp_solver = VirtualPacketSolver(
        montecarlo_configuration,
        spectrum_frequency_grid,
        geometry_state_numba,
        time_explosion,
    )
    
    # Generate virtual packets from tracker spawn events
    virtual_packet_state = vp_solver.generate_virtual_packets(
        tracker_df,
        NUMBER_OF_VPACKETS,
    )
    
    print(f"Generated {len(virtual_packet_state.nus):,} virtual packets")
    print(f"Frequency range: {virtual_packet_state.nus.min():.2e} - {virtual_packet_state.nus.max():.2e} Hz")
    print(f"Total energy: {virtual_packet_state.energies.sum():.2e} erg")
else:
    print("Virtual packet generation requires ENABLE_RPACKET_TRACKING=True and NUMBER_OF_VPACKETS > 0")

### Virtual Packet State Properties

The `VirtualPacketState` object provides access to all virtual packet properties:

In [ ]:
if ENABLE_RPACKET_TRACKING and NUMBER_OF_VPACKETS > 0:
    print("Available properties:")
    print(f"  nus: {len(virtual_packet_state.nus)} frequencies")
    print(f"  energies: {len(virtual_packet_state.energies)} energy values")  
    print(f"  initial_rs: {len(virtual_packet_state.initial_rs)} radii")
    print(f"  initial_mus: {len(virtual_packet_state.initial_mus)} propagation angles")
    print(f"  last_interaction_type: {len(virtual_packet_state.last_interaction_type)} interaction codes")

## Visualizing Virtual Packet Distribution

In [ ]:
if ENABLE_RPACKET_TRACKING and NUMBER_OF_VPACKETS > 0:
    # Convert to wavelengths
    wavelengths = (virtual_packet_state.nus * u.Hz).to(u.AA, equivalencies=u.spectral())
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Wavelength distribution
    axes[0].hist(wavelengths.value, bins=50, alpha=0.7, edgecolor='black')
    axes[0].set_xlabel(r"Wavelength [$\AA$]")
    axes[0].set_ylabel("Number of Virtual Packets")
    axes[0].set_title("Virtual Packet Wavelength Distribution")
    axes[0].grid(alpha=0.3)
    
    # Energy distribution
    axes[1].hist(np.log10(virtual_packet_state.energies), bins=50, alpha=0.7, edgecolor='black', color='orange')
    axes[1].set_xlabel(r"$\log_{10}$(Energy [erg])")
    axes[1].set_ylabel("Number of Virtual Packets")
    axes[1].set_title("Virtual Packet Energy Distribution")
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Comparing Spawn Event Types

Examine the distribution of spawn events by interaction type:

In [ ]:
if ENABLE_RPACKET_TRACKING:
    print("Spawn event statistics:")
    print(f"  Total events: {len(tracker_df):,}")
    print(f"  Unique packets: {tracker_df['id'].nunique():,}")
    print(f"  Mean events per packet: {len(tracker_df)/tracker_df['id'].nunique():.2f}")
    
    # Status distribution
    print(f"\nStatus distribution:")
    print(tracker_df['status'].value_counts())

## Summary

**Key Points:**
- Run transport with full tracking enabled to capture spawn events  
- Tracker data stored in `rpacket_trackers`, convert to DataFrame with `trackers_full_to_df()`
- Generate virtual packets via postprocessing with `VirtualPacketSolver`
- Virtual packet state provides access to all packet properties for analysis
- Postprocessing architecture enables flexible spectrum generation from tracker data

For more details, see:
- [Virtual Packets Physics](../physics_walkthrough/spectrum/virtualpackets.rst)
- [Virtual Packet Postprocessing Guide](../analyzing_tardis/spectrum/virtual_packet_postprocessing.ipynb)